# 01 - Exploratory Data Analysis

**Goal of this notebook:**
- Mount Google Drive and set up the project structure
- Download the LIDC-IDRI (lung CT) dataset
- Analyze class distribution, image sizes, and pixel statistics
- Generate all EDA figures for the report
- Produce stratified train/val/test splits saved as CSV

### Environment Setup



In [ ]:
# Configuration des chemins locaux
import os

# Modifiez ROOT selon votre structure locale
ROOT      = os.environ.get('PROJECT_ROOT') or (os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()).lower() == 'notebooks' else os.getcwd())   # ou un chemin absolu, ex: '/home/user/PFA'
os.environ['PROJECT_ROOT'] = ROOT
DATA_RAW  = os.path.join(ROOT, 'data', 'raw')
DATA_PROC = os.path.join(ROOT, 'data', 'processed')
FIGURES   = os.path.join(ROOT, 'results', 'figures', 'eda')

for d in [DATA_RAW, DATA_PROC, FIGURES]:
    os.makedirs(d, exist_ok=True)

print('Project structure ready:')
for d in [ROOT, DATA_RAW, DATA_PROC, FIGURES]:
    print(f'  {d}')


In [ ]:
# Installation des dépendances
import subprocess, sys
pkgs = ['kaggle', 'timm', 'einops', 'matplotlib', 'seaborn',
        'scikit-learn', 'Pillow', 'tqdm', 'pandas', 'numpy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Installation terminée.')


In [ ]:
# Les chemins ROOT, DATA_RAW, DATA_PROC, FIGURES sont déjà définis ci-dessus.
print('Chemins configurés :', ROOT)


### 1. Dataset Download (LIDC-IDRI via Kaggle)


In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = "KGAT_aff9002e33d1adb0f63b7ec4826eb8ee"

print("Kaggle API Token configured successfully!")


In [ ]:
# Ce dataset (LIDC-IDRI segmentation) a été abandonné au profit du dataset ci-dessous.
# Vous pouvez ignorer cette cellule.

# import os
# os.chdir(DATA_RAW)
# !kaggle datasets download -d kmader/finding-lungs-in-ct-data --unzip


### 2. Imports & Configuration

In [ ]:
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F7F9FC',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

PALETTE = ['#0A7E8C', '#D95F2B', '#1A7A4A', '#C47D0E', '#5A3E8C']
CLASS_NAMES = {0: 'No Nodule', 1: 'Nodule'}

print("Imports OK | Seed:", SEED)

### 3. Dataset Loading & Structure Inspection


In [ ]:
import os
print(os.listdir(DATA_RAW))

In [ ]:
all_image_paths = []

for ext in ('*.tif', '*.tiff'):
    all_image_paths.extend(glob.glob(os.path.join(DATA_RAW, '**', ext), recursive=True))

print(f"Total images found: {len(all_image_paths)}")
print("\nSample paths:")
for p in all_image_paths[:5]:
    print(f"  {p}")

In [ ]:
records = []
for path in all_image_paths:
    label = os.path.basename(os.path.dirname(path))
    records.append({'path': path, 'label': label})

df = pd.DataFrame(records)
print("Raw label counts:")
print(df['label'].value_counts())

In [ ]:
import pandas as pd
import os

# 1. Define the exact path to the CSV file
csv_path = os.path.join(DATA_RAW, 'lung_stats.csv')

# 2. Load it into a Pandas DataFrame
stats_df = pd.read_csv(csv_path)

# 3. Print the column names and show the first 5 rows
print("CSV Columns found:")
print(stats_df.columns.tolist())
print("-" * 50)
display(stats_df.head())

In [ ]:
!rm -rf 2d_images 2d_masks 3d_images lung_stats.csv *.zip
print("Old data cleared!")

###EDA Data Pivot: Segmentation vs. Classification

**The Discovery:** During our initial data loading phase, we discovered that our original Kaggle dataset was engineered for **image segmentation**, not binary classification. The directories (`2d_images` and `2d_masks`) contained raw CT scans and their corresponding black-and-white lung boundary silhouettes, completely lacking any clinical diagnosis labels.

**The Implication:** If we proceeded to train a ResNet50 on this structure, the model would simply learn to differentiate a raw medical scan from a drawn mask—which mathematically makes zero sense and holds no clinical value for disease detection.

**The Pivot:** To achieve our architectural goal, we are swapping the dataset for a dedicated binary classification source (`ucimachinelearning/lung-nodule-dataset`). This new dataset is correctly pre-organized into distinct clinical classes (`Healthy` vs. `Lung_Nodule`), allowing our network to learn the actual morphological features of pulmonary nodules.

*Note: This pivot perfectly validates the necessity of conducting rigorous Exploratory Data Analysis (EDA) before initiating any deep learning training loops.*

###REDO WITH NEW DATASET

### 1. Dataset Download

In [ ]:
import os, subprocess

os.chdir(DATA_RAW)

# Téléchargement via l'API Kaggle (nécessite les credentials ci-dessus)
subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'ucimachinelearning/lung-nodule-dataset', '--unzip'],
    check=True
)

print('\nFichiers dans DATA_RAW :')
print(os.listdir(DATA_RAW)[:10])


### 2. Imports & Configuration

In [ ]:
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F7F9FC',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

PALETTE = ['#0A7E8C', '#D95F2B', '#1A7A4A', '#C47D0E', '#5A3E8C']
CLASS_NAMES = {0: 'No Nodule', 1: 'Nodule'}

print("Imports OK | Seed:", SEED)

### 3. Dataset Loading & Structure Inspection


In [ ]:
import os
print(os.listdir(DATA_RAW))

In [ ]:
all_image_paths = []
for ext in ('*.png', '*.jpg', '*.jpeg'):
    all_image_paths.extend(glob.glob(os.path.join(DATA_RAW, '**', ext), recursive=True))

print(f"Total images found: {len(all_image_paths)}")
print("\nSample paths:")
for p in all_image_paths[:5]:
    print(f"  {p}")

In [ ]:
records = []
for path in all_image_paths:
    label = os.path.basename(os.path.dirname(path))
    records.append({'path': path, 'label': label})

df = pd.DataFrame(records)
print("Raw label counts:")
print(df['label'].value_counts())

In [ ]:
LABEL_MAP = {
    'Lung_Nodule': 1,
    'Healthy': 0
}

df['label_raw'] = df['label']
df['label']     = df['label_raw'].map(LABEL_MAP)

n_before = len(df)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"Dropped {n_before - len(df)} rows with unrecognized labels.")
print(f"Final: class 0={int((df['label']==0).sum())}  class 1={int((df['label']==1).sum())}")
df.head()

### 4. Class Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts().sort_index()
labels_str = [CLASS_NAMES[i] for i in counts.index]

# Bar
bars = axes[0].bar(labels_str, counts.values, color=PALETTE[:2], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of Images')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 10, f'{val:,}', ha='center', fontweight='bold')

# Pie
axes[1].pie(counts.values, labels=labels_str, autopct='%1.1f%%',
            colors=PALETTE[:2], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Balance')

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

ratio = counts.max() / counts.min()
print(f"Imbalance ratio: {ratio:.2f}:1")
if ratio > 2:
    print("WARNING: Imbalanced dataset. Will apply class weights during training.")

### 5. Sample Image Gallery


In [ ]:
def show_samples(df, label, label_name, n=8, cols=8):
    subset = df[df['label'] == label]
    paths  = subset['path'].sample(n=min(n, len(subset)), random_state=SEED).tolist()
    rows   = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 1.9))
    axes = np.array(axes).flatten()
    for ax, path in zip(axes, paths):
        img = Image.open(path).convert('RGB')
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    for ax in axes[len(paths):]:
        ax.axis('off')
    fig.suptitle(f'Class: {label_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES, f'samples_class_{label}.png'), dpi=150, bbox_inches='tight')
    plt.show()

show_samples(df, label=1, label_name='Nodule (Positive)')
show_samples(df, label=0, label_name='No Nodule (Negative)')

### 6. Image Size & Pixel Statistics


In [ ]:
SAMPLE_N = min(500, len(df))
sample_df = df.sample(n=SAMPLE_N, random_state=SEED)

widths, heights, means, stds = [], [], [], []

for _, row in tqdm(sample_df.iterrows(), total=SAMPLE_N, desc='Reading images'):
    try:
        img = np.array(Image.open(row['path']).convert('RGB'), dtype=np.float32) / 255.0
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)
        means.append(float(img.mean()))
        stds.append(float(img.std()))
    except Exception as e:
        print(f"Error: {e}")

print(f"Width  - min:{min(widths)} max:{max(widths)} mean:{np.mean(widths):.1f}")
print(f"Height - min:{min(heights)} max:{max(heights)} mean:{np.mean(heights):.1f}")
print(f"Pixel mean: {np.mean(means):.4f}")
print(f"Pixel std : {np.mean(stds):.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, data, title, color in [
    (axes[0,0], widths,  'Image Width Distribution',  PALETTE[0]),
    (axes[0,1], heights, 'Image Height Distribution', PALETTE[2]),
    (axes[1,0], means,   'Per-Image Pixel Mean',      PALETTE[3]),
    (axes[1,1], stds,    'Per-Image Pixel Std',       PALETTE[4]),
]:
    ax.hist(data, bins=35, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(data), color=PALETTE[1], lw=2, linestyle='--',
               label=f'Mean={np.mean(data):.2f}')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Image Size & Pixel Statistics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'image_statistics.png'), dpi=150, bbox_inches='tight')
plt.show()

### 7. Per-Class Pixel Intensity Distribution


In [ ]:
class_pixels = {0: [], 1: []}

for label in [0, 1]:
    cls_df = df[df['label'] == label].sample(n=min(200, int((df['label']==label).sum())), random_state=SEED)
    for _, row in tqdm(cls_df.iterrows(), desc=f'Class {label}', total=len(cls_df)):
        try:
            img = np.array(Image.open(row['path']).convert('L'), dtype=np.float32) / 255.0
            class_pixels[label].extend(img.flatten().tolist())
        except:
            pass

fig, ax = plt.subplots(figsize=(11, 5))
for label, color in [(0, PALETTE[0]), (1, PALETTE[1])]:
    px = class_pixels[label]
    ax.hist(px, bins=80, alpha=0.6, color=color,
            label=f'{CLASS_NAMES[label]}  (mean={np.mean(px):.3f})', density=True)

ax.set_title('Pixel Intensity Distribution by Class (Grayscale)')
ax.set_xlabel('Pixel value [0, 1]')
ax.set_ylabel('Density')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'pixel_intensity_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8. Feature Correlation (Image Metadata)


In [ ]:
meta_df = sample_df.copy()
meta_df['width']  = widths
meta_df['height'] = heights
meta_df['mean']   = means
meta_df['std']    = stds
meta_df['aspect'] = meta_df['width'] / meta_df['height']

fig, ax = plt.subplots(figsize=(8, 5))

features = meta_df[['label', 'width', 'height', 'mean', 'std', 'aspect']]
dynamic_features = features.loc[:, features.nunique() > 1]

corr = dynamic_features.corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (Cleaned)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'correlation_heatmap_cleaned.png'), dpi=150, bbox_inches='tight')
plt.show()

### 9. Stratified Train / Val / Test Split


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    c0 = int((sdf['label']==0).sum())
    c1 = int((sdf['label']==1).sum())
    print(f"{name:6}: total={len(sdf):5}  no_nodule={c0}  nodule={c1}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name, sdf in zip(axes, ['Train', 'Validation', 'Test'], [train_df, val_df, test_df]):
    counts = sdf['label'].value_counts().sort_index()
    bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values,
                  color=PALETTE[:2], edgecolor='white', width=0.5)
    ax.set_title(f'{name}  (n={len(sdf)})')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                str(val), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'split_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train_df.to_csv(os.path.join(DATA_PROC, 'train.csv'), index=False)
val_df.to_csv(  os.path.join(DATA_PROC, 'val.csv'),   index=False)
test_df.to_csv( os.path.join(DATA_PROC, 'test.csv'),  index=False)
print("Splits saved to Drive.")

### 10. Class Weights & EDA Summary


In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)
pos_weight = class_weights[1] / class_weights[0]

print("Class weights:")
print(f"  class 0 (No Nodule): {class_weights[0]:.4f}")
print(f"  class 1 (Nodule)   : {class_weights[1]:.4f}")
print(f"For CrossEntropyLoss (linear probe):")
print(f"  class_weights = torch.tensor([{class_weights[0]:.4f}, {class_weights[1]:.4f}])")

In [ ]:
total     = len(df)
n_nod     = int((df['label']==1).sum())
n_no      = int((df['label']==0).sum())
ratio_val = n_no / n_nod if n_nod > 0 else 0

lines = [
    "=" * 42,
    "  EDA SUMMARY",
    "=" * 42,
    f"Total images      : {total}",
    f"  Nodule (1)      : {n_nod}  ({100*n_nod/total:.1f}%)",
    f"  No Nodule (0)   : {n_no}  ({100*n_no/total:.1f}%)",
    f"Imbalance ratio   : {ratio_val:.2f}:1",
    f"Mean image size   : {int(np.mean(widths))}x{int(np.mean(heights))} px",
    f"Global pixel mean : {np.mean(means):.4f}",
    f"Global pixel std  : {np.mean(stds):.4f}",
    f"Train / Val / Test: {len(train_df)} / {len(val_df)} / {len(test_df)}",
    f"pos_weight        : {pos_weight:.4f}",
    f"  class_weights = torch.tensor([{class_weights[0]:.4f}, {class_weights[1]:.4f}])"
    "=" * 42,
    f"Pre-train pool     : {total} images (unlabelled, all splits)",
]
summary_text = "\n".join(lines)
print(summary_text)

with open(os.path.join(FIGURES, 'eda_summary.txt'), 'w') as f:
    f.write(summary_text)
print("\nSummary saved.")